# Extragere date FBref

Notebook-ul preia statistici la nivel de jucător și meci din FBref: minute, goluri, pase decisive, șuturi și cartonașe, în măsura în care apar în tabelul FBref. Datele se salvează ca CSV. Notebook-ul nu curăță datele și nu antrenează modele.

## 1. Importuri și setări

Se folosește biblioteca `soccerdata`, care citește tabelele FBref și le pune în DataFrame-uri pandas.

In [ ]:
from pathlib import Path
import pandas as pd
import soccerdata as sd

OUTPUT_DIR = Path("data/raw/fbref")
CACHE_DIR = Path("data/cache/fbref")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LEAGUES = ["ENG-Premier League"]
SEASONS = ["2023-2024", "2024-2025"]

print("Ligi:", LEAGUES)
print("Sezoane:", SEASONS)
print("Output:", OUTPUT_DIR.resolve())


## 2. Inițializarea cititorului FBref

ID-urile uzuale pentru ligile de top includ `ENG-Premier League`, `ESP-La Liga`, `GER-Bundesliga`, `ITA-Serie A` și `FRA-Ligue 1`. Schimbă lista după nevoie.

In [ ]:
fbref = sd.FBref(
    leagues=LEAGUES,
    seasons=SEASONS,
    data_dir=CACHE_DIR,
)


## 3. Extragerea jurnalelor de meci pentru jucători

Tabelul `summary` include câmpuri precum minute (`Min`), goluri (`Gls`), pase decisive (`Ast`), șuturi (`Sh`), șuturi pe poartă (`SoT`) și cartonașe (`CrdY`, `CrdR`), dacă sunt disponibile în sursă.

In [ ]:
player_match_stats = fbref.read_player_match_stats(stat_type="summary")

if player_match_stats.empty:
    raise RuntimeError("FBref nu a returnat rânduri pentru ligile și sezoanele selectate.")

player_match_stats.head()


## 4. Salvarea datelor extrase

Se păstrează și indexul MultiIndex ca și coloane, pentru a nu pierde identificatorii ligi/sezonului/meciului/jucătorului.

In [ ]:
player_match_stats_export = player_match_stats.reset_index()
output_file = OUTPUT_DIR / "fbref_player_match_stats.csv"
player_match_stats_export.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Salvat {len(player_match_stats_export):,} rânduri în {output_file.resolve()}")


## 5. (Opțional) Extragerea programului și rezultatelor meciurilor

In [ ]:
schedule = fbref.read_schedule()
schedule_export = schedule.reset_index()
schedule_file = OUTPUT_DIR / "fbref_schedule.csv"
schedule_export.to_csv(schedule_file, index=False, encoding="utf-8-sig")
print(f"Salvate {len(schedule_export):,} meciuri în {schedule_file.resolve()}")
schedule_export.head()


## 6. Verificare rapidă

Verifică numărul de valori lipsă și coloanele disponibile înainte de etapa de curățare.

In [ ]:
print("Dimensiune:", player_match_stats_export.shape)
display(player_match_stats_export.dtypes.to_frame("tip"))
display(player_match_stats_export.isna().sum().sort_values(ascending=False).head(20).to_frame("valori_lipsa"))


### Limitare de acces

FBref poate respinge cererile automate cu HTTP 403 sau poate limita accesul. Dacă se întâmplă, notebook-ul se va opri cu eroarea primită de la site; reîncearcă mai târziu sau folosește exportul manual oferit de FBref. `soccerdata` păstrează cache local pentru a evita descărcări repetate. Nu rula notebook-ul repetat într-un interval scurt.